# Experiment G: 100,000 images, 5 epochs
Fine-tune one student on **100,000 COCO train2017 images for 5 epochs**: the same number of training steps as
Experiment F (5,000 images × 100 epochs), but 20× more different images. Run this notebook on **three Colab
accounts at the same time**, one per encoder start: set `INIT` in Cell 1 to `'sami'`, `'mae'` or `'random'`.

| Cell | What it does | Time |
|---|---|---|
| 1 | GPU check, Drive, code | ~2 min |
| 2 | Download 100,000 images and build the training data on this machine's disk (not on Drive) | ~30–60 min |
| 3 | Training, saves to the shared Drive folder every 1,000 steps (~4 min) | ~3.5–4.5 h |
| 4 | Evaluation on our 1,620 held-out objects, downloads the CSV | ~5 min |

**If Colab disconnects or an account runs out of GPU time:** open this notebook on the same or another account
(it needs the `effsam_work` shortcut in *My Drive*), keep the same `INIT`, and run Cells 1, 2, 3 again.
Cell 2 rebuilds exactly the same data (fixed seed, checked automatically), and Cell 3 continues from the last
save, losing at most ~4 minutes of training.

In [ ]:
INIT = 'sami'          # <-- set per account: 'sami', 'mae' or 'random'

import os, shutil, subprocess, base64, json, torch
os.chdir('/content')
assert INIT in ('sami', 'mae', 'random')
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU (or this account has no GPU time left).'

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/effsam_work'
assert os.path.exists(f'{WORK}/features_5k/subset_instances_train2017.json'), \
    'effsam_work not found: add the shared folder to My Drive (Organize -> Add shortcut -> My Drive)'
ENC = {'sami': f'{WORK}/checkpoints/sami/encoder_final.pt', 'mae': f'{WORK}/checkpoints/mae/encoder_final.pt', 'random': None}
assert ENC[INIT] is None or os.path.exists(ENC[INIT]), f'encoder not found: {ENC[INIT]}'
free = shutil.disk_usage('/content').free / 1e9
print(f'free local disk: {free:.0f} GB')
assert free > 35, 'need ~30 GB of local disk for the data'

DATA = '/content/data/COCO_dataset'                     # our subset: needed for the evaluation images
if not os.path.exists(DATA):
    shutil.copytree('/content/drive/MyDrive/COCO_dataset', DATA)

CODE = '/content/EfficientSAM'
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', 'coco-eda-sharing',
                'https://github.com/Mehdi-blip-stack/EfficientSAM', CODE], check=True)
# Experiment G code (finetune.py with --large / --save_steps, prepare_100k.py), tested before sharing
FILES = {"finetune.py": "...", "prepare_100k.py": "IiIiRXhwZXJpbWVudCBHLCBzdGVwIDE6IGJ1aWxkIGEgMTAwLDAwMC1pbWFnZSBmaW5lLXR1bmluZyBzZXQgZnJvbSBDT0NPIHRyYWluMjAxNy4gV3JpdHRlbiBieSBvdXIgdGVhbS4KClRoZSBzZXQgaXMgb3VyIDUsMDAwLWltYWdlIHNldCBvZiBFeHBlcmltZW50IEUgKHdoaWNoIGNvbnRhaW5zIG91ciBvcmlnaW5hbCA4MDAgdHJhaW5pbmcgaW1hZ2VzKSBwbHVzIGltYWdlcwpkcmF3biBhdCByYW5kb20gKGZpeGVkIHNlZWQpIGZyb20gdGhlIHJlc3Qgb2YgQ09DTyB0cmFpbjIwMTcsIHNvIDgwMCA8IDUsMDAwIDwgMTAwLDAwMCBhcmUgbmVzdGVkIGFuZCBub25lIG9mCnRoZW0gb3ZlcmxhcHMgb3VyIDIwMCBoZWxkLW91dCB2YWwyMDE3IGV2YWx1YXRpb24gaW1hZ2VzLiBPbmx5IGZpbmUtdHVuaW5nIHVzZXMgaXQ6IHRoZSBTQU1JIGFuZCBNQUUgZW5jb2RlcnMKc3RheSB0aGUgb25lcyBwcmV0cmFpbmVkIG9uIHRoZSBvcmlnaW5hbCA4MDAgaW1hZ2VzLgoKV2h5IGEgc2VwYXJhdGUgc2NyaXB0IChwcmVwYXJlXzVrLnB5IGJ1aWxkcyB0aGUgNSwwMDAtaW1hZ2Ugc2V0KTogMTAwLDAwMCBpbWFnZXMgZG8gbm90IGZpdCBpbiBDb2xhYidzIG1lbW9yeQp0aGUgd2F5IDUsMDAwIGRvIChhbGwgaW1hZ2VzICsgdGhlaXIgbWlycm9ycyB3b3VsZCBiZSB+NDAgR0IsIGFsbCBtYXNrcyB1bnBhY2tlZCB+NDggR0IpLiBUaGlzIHNjcmlwdAogICogc3RyZWFtcyBldmVyeSBpbWFnZSBmcm9tIGltYWdlcy5jb2NvZGF0YXNldC5vcmcgc3RyYWlnaHQgaW50byBhbiBvbi1kaXNrIGFycmF5IGF0IDI1NiB4IDI1NiAobm8gcmF3IGltYWdlIGlzCiAgICBrZXB0KSwgV0lUSE9VVCB0aGUgbWlycm9yZWQgY29weTogZmluZXR1bmUucHkgLS1sYXJnZSBmbGlwcyB0aGUgaW1hZ2Ugb24gdGhlIGZseSwgd2hpY2ggZ2l2ZXMgZXhhY3RseSB0aGUgc2FtZQogICAgcGl4ZWxzIGJlY2F1c2UgdGhlIGJpbGluZWFyIHJlc2l6ZSBpcyBzeW1tZXRyaWM7CiAgKiBzdG9yZXMgdGhlIG1hc2tzIGJpdC1wYWNrZWQgaW4gYW4gdW5jb21wcmVzc2VkIG9uLWRpc2sgYXJyYXksIHNvIGZpbmV0dW5lLnB5IC0tbGFyZ2UgcmVhZHMgb25seSB0aGUgbWFza3Mgb2YKICAgIHRoZSBjdXJyZW50IGJhdGNoLgpJbnB1dHMsIG1hc2tzIGFuZCBib3hlcyBhcmUgbWFkZSBleGFjdGx5IGFzIGluIHByZXBhcmVfNWsucHkgLyBmaW5ldHVuZS5idWlsZF90YXJnZXRzIChzYW1lIHNxdWFzaCgpIHJlc2l6ZSwKc2FtZSBtYXNrIHJ1bGVzOiBub24tY3Jvd2QsID49IDQgcHgsIHN0aWxsIHZpc2libGUgYXQgMjU2IHB4KS4KCk91dHB1dHMgaW4gLS1vdXQgKGxvY2FsIGRpc2s7IH4yMCBHQiBpbnB1dHMgKyB+NiBHQiBtYXNrcyk6CiAgaW1hZ2VfaWRzLm5weSwgc3R1ZGVudF9pbnB1dHNfMjU2X3NpbmdsZS5ucHkgKE4sMywyNTYsMjU2IHVpbnQ4KSwgdGFyZ2V0c19yb3dzLm5weSAoSyw1IGludDMyOiBpbWFnZSBpbmRleCwKICB4MCwgeTAsIHgxLCB5MSksIHRhcmdldHNfbWFza3MubnB5ICg+PUssIDgxOTIgdWludDgsIGJpdC1wYWNrZWQgMjU2eDI1NiksIHByZXBhcmVfMTAwa19pbmZvLmpzb24KRXZlcnkgc3RhZ2UgaXMgc2tpcHBlZC9yZXN1bWVkIGlmIGl0cyBvdXRwdXQgZXhpc3RzLCBzbyByZS1ydW5uaW5nIGFmdGVyIGEgZGlzY29ubmVjdCBjb250aW51ZXMuCkRldGVybWluaXN0aWM6IHRoZSBzYW1lIHNlZWQgZ2l2ZXMgdGhlIHNhbWUgaW1hZ2VzLCBtYXNrcyBhbmQgb3JkZXIgb24gZXZlcnkgbWFjaGluZSAoY2hlY2tlZCBieSBhIGZpbmdlcnByaW50CmluIHByZXBhcmVfMTAwa19pbmZvLmpzb24sIHdoaWNoIGZpbmV0dW5lLnB5IC0tbGFyZ2Ugc3RvcmVzIGluIGl0cyBjaGVja3BvaW50KS4KClVzYWdlOgogIHB5dGhvbiAtbSByZXByby5wcmVwYXJlXzEwMGsgLS1iYXNlX3N1YnNldCAuLi4vZWZmc2FtX3dvcmsvZmVhdHVyZXNfNWsvc3Vic2V0X2luc3RhbmNlc190cmFpbjIwMTcuanNvbiBcCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAtLW91dCAvY29udGVudC9mZWF0dXJlc18xMDBrCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCB0aHJlYWRpbmcKaW1wb3J0IHRpbWUKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CmltcG9ydCB6aXBmaWxlCmZyb20gY29uY3VycmVudC5mdXR1cmVzIGltcG9ydCBUaHJlYWRQb29sRXhlY3V0b3IKZnJvbSBtdWx0aXByb2Nlc3NpbmcgaW1wb3J0IFBvb2wKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpmcm9tIC5jb21tb24gaW1wb3J0IE1JTl9NQVNLX1BJWEVMUywgU1RVREVOVF9TSVpFLCBkZWNvZGUsIHNxdWFzaAoKUyA9IFNUVURFTlRfU0laRQpQQUNLRUQgPSBTICogUyAvLyA4CkFOTl9VUkwgPSAiaHR0cDovL2ltYWdlcy5jb2NvZGF0YXNldC5vcmcvYW5ub3RhdGlvbnMvYW5ub3RhdGlvbnNfdHJhaW52YWwyMDE3LnppcCIKCgpkZWYgZmV0Y2hfYnl0ZXModXJsLCB0cmllcz02KToKICAgIGZvciBrIGluIHJhbmdlKHRyaWVzKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3Blbih1cmwsIHRpbWVvdXQ9NjApIGFzIHI6CiAgICAgICAgICAgICAgICByZXR1cm4gci5yZWFkKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIG5ldHdvcmsgaGljY3VwOiB3YWl0IGFuZCByZXRyeQogICAgICAgICAgICBpZiBrID09IHRyaWVzIC0gMToKICAgICAgICAgICAgICAgIHByaW50KCIgIGZhaWxlZDoiLCB1cmwsIGUsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICB0aW1lLnNsZWVwKDIgKiAoayArIDEpKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1hc2tzICh3b3JrZXIpCkcgPSB7fSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgZmlsbGVkIGJlZm9yZSB0aGUgcG9vbCBmb3JrcyAoY29weS1vbi13cml0ZSkKCgpkZWYgbWFza3NfZm9yX2NodW5rKGxvX2hpKToKICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcygxKQogICAgbG8sIGhpID0gbG9faGkKICAgIHJvd3MsIHBhY2tlZCA9IFtdLCBbXQogICAgZm9yIGkgaW4gcmFuZ2UobG8sIGhpKToKICAgICAgICBoLCB3LCBhbm5zID0gR1siaW1ncyJdW2ldCiAgICAgICAgZm9yIGFubiBpbiBhbm5zOgogICAgICAgICAgICBtID0gZGVjb2RlKGFubiwgaCwgdykKICAgICAgICAgICAgaWYgbS5zdW0oKSA8IE1JTl9NQVNLX1BJWEVMUzoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIG0yNTYgPSBGLmludGVycG9sYXRlKHRvcmNoLmZyb21fbnVtcHkobSlbTm9uZSwgTm9uZV0uZmxvYXQoKSwgKFMsIFMpLCBtb2RlPSJiaWxpbmVhciIpWzAsIDBdID4gMC41CiAgICAgICAgICAgIGlmIG0yNTYuc3VtKCkgPT0gMDogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyB0aGlubmVyIHRoYW4gb25lIHBpeGVsIGFmdGVyIHJlc2l6aW5nCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB5cywgeHMgPSB0b3JjaC53aGVyZShtMjU2KQogICAgICAgICAgICByb3dzLmFwcGVuZChbaSwgeHMubWluKCkuaXRlbSgpLCB5cy5taW4oKS5pdGVtKCksIHhzLm1heCgpLml0ZW0oKSArIDEsIHlzLm1heCgpLml0ZW0oKSArIDFdKQogICAgICAgICAgICBwYWNrZWQuYXBwZW5kKG5wLnBhY2tiaXRzKG0yNTYubnVtcHkoKS5yZXNoYXBlKC0xKSkpCiAgICByZXR1cm4gKG5wLmFycmF5KHJvd3MsIG5wLmludDMyKS5yZXNoYXBlKC0xLCA1KSwKICAgICAgICAgICAgbnAuc3RhY2socGFja2VkKSBpZiBwYWNrZWQgZWxzZSBucC56ZXJvcygoMCwgUEFDS0VEKSwgbnAudWludDgpKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXNlX3N1YnNldCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9InN1YnNldF9pbnN0YW5jZXNfdHJhaW4yMDE3Lmpzb24gb2YgdGhlIDUsMDAwLWltYWdlIHNldCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0ibG9jYWwgZm9sZGVyIGZvciB0aGUgYXJyYXlzICh+MjYgR0IpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAwMDAwKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0zNjIyMikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10aHJlYWRzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDgsIGhlbHA9ImRvd25sb2FkIHRocmVhZHMiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD1vcy5jcHVfY291bnQoKSBvciAyLCBoZWxwPSJwcm9jZXNzZXMgZm9yIHRoZSBtYXNrcyIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBvdXQgPSBQYXRoKGFyZ3Mub3V0KTsgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICAjIDEuIGZ1bGwgdHJhaW4yMDE3IGFubm90YXRpb25zICh+MjUwIE1CIHppcCkKICAgIGZ1bGwgPSBvdXQgLyAicmF3IiAvICJhbm5vdGF0aW9ucyIgLyAiaW5zdGFuY2VzX3RyYWluMjAxNy5qc29uIgogICAgaWYgbm90IGZ1bGwuZXhpc3RzKCk6CiAgICAgICAgKG91dCAvICJyYXciKS5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgICAgIHogPSBvdXQgLyAicmF3IiAvICJhbm5vdGF0aW9uc190cmFpbnZhbDIwMTcuemlwIgogICAgICAgIGlmIG5vdCB6LmV4aXN0cygpOgogICAgICAgICAgICBwcmludCgiZG93bmxvYWRpbmcgQ09DTyBhbm5vdGF0aW9ucyAofjI1MCBNQikgLi4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgZGF0YSA9IGZldGNoX2J5dGVzKEFOTl9VUkwpOyBhc3NlcnQgZGF0YSwgImNvdWxkIG5vdCBkb3dubG9hZCB0aGUgQ09DTyBhbm5vdGF0aW9ucyIKICAgICAgICAgICAgei53cml0ZV9ieXRlcyhkYXRhKTsgZGVsIGRhdGEKICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh6KSBhcyBmOgogICAgICAgICAgICBmLmV4dHJhY3QoImFubm90YXRpb25zL2luc3RhbmNlc190cmFpbjIwMTcuanNvbiIsIG91dCAvICJyYXciKQogICAgcHJpbnQoZiJsb2FkaW5nIGFubm90YXRpb25zIC4uLiAoe3RpbWUudGltZSgpIC0gdDA6LjBmfXMpIiwgZmx1c2g9VHJ1ZSkKICAgIEQgPSBqc29uLmxvYWQob3BlbihmdWxsKSkKCiAgICAjIDIuIHN1YnNldDogdGhlIDUsMDAwLWltYWdlIHNldCArIHJhbmRvbSBvdGhlcnMgKHNvcnRlZCBieSBpbWFnZSBpZCwgbGlrZSB0aGUgb3RoZXIgZmVhdHVyZSBzZXRzKQogICAgYmFzZSA9IHtpbVsiaWQiXSBmb3IgaW0gaW4ganNvbi5sb2FkKG9wZW4oYXJncy5iYXNlX3N1YnNldCkpWyJpbWFnZXMiXX0KICAgIG90aGVycyA9IHNvcnRlZChpbVsiaWQiXSBmb3IgaW0gaW4gRFsiaW1hZ2VzIl0gaWYgaW1bImlkIl0gbm90IGluIGJhc2UpCiAgICByYW5kb20uUmFuZG9tKGFyZ3Muc2VlZCkuc2h1ZmZsZShvdGhlcnMpCiAgICBrZWVwID0gYmFzZSB8IHNldChvdGhlcnNbOiBtYXgoMCwgYXJncy5uIC0gbGVuKGJhc2UpKV0pCiAgICBpbWFnZXMgPSBzb3J0ZWQoW2ltIGZvciBpbSBpbiBEWyJpbWFnZXMiXSBpZiBpbVsiaWQiXSBpbiBrZWVwXSwga2V5PWxhbWJkYSByOiByWyJpZCJdKQogICAgaWRzID0gbnAuYXJyYXkoW2ltWyJpZCJdIGZvciBpbSBpbiBpbWFnZXNdLCBucC5pbnQ2NCkKICAgIGlmIChvdXQgLyAiaW1hZ2VfaWRzLm5weSIpLmV4aXN0cygpOgogICAgICAgIGFzc2VydCBucC5hcnJheV9lcXVhbChucC5sb2FkKG91dCAvICJpbWFnZV9pZHMubnB5IiksIGlkcyksICJpbWFnZSBsaXN0IGRpZmZlcnMgZnJvbSB0aGUgb25lIG9uIGRpc2siCiAgICBucC5zYXZlKG91dCAvICJpbWFnZV9pZHMubnB5IiwgaWRzKQogICAgbiA9IGxlbihpbWFnZXMpCiAgICBwcmludChmInN1YnNldDoge259IGltYWdlcyAoe2xlbihiYXNlKX0gZnJvbSB0aGUgNSwwMDAtaW1hZ2Ugc2V0KSAoe3RpbWUudGltZSgpIC0gdDA6LjBmfXMpIiwgZmx1c2g9VHJ1ZSkKCiAgICAjIDMuIHN0dWRlbnQgaW5wdXRzOiBzdHJlYW0sIHNxdWFzaCB0byAyNTYgeCAyNTYsIHdyaXRlOyBkb25lIGZsYWdzIG1ha2UgaXQgcmVzdW1hYmxlCiAgICBpbnAgPSBvdXQgLyAic3R1ZGVudF9pbnB1dHNfMjU2X3NpbmdsZS5ucHkiCiAgICBmbGFnc19wID0gb3V0IC8gImlucHV0c19kb25lLm5weSIKICAgIGlmIG5vdCAob3V0IC8gImlucHV0c19jb21wbGV0ZSIpLmV4aXN0cygpOgogICAgICAgIGFyciA9IG5wLmxpYi5mb3JtYXQub3Blbl9tZW1tYXAoc3RyKGlucCksIG1vZGU9InIrIiBpZiBpbnAuZXhpc3RzKCkgZWxzZSAidysiLCBkdHlwZT1ucC51aW50OCwgc2hhcGU9KG4sIDMsIFMsIFMpKQogICAgICAgIGZsYWdzID0gbnAubGliLmZvcm1hdC5vcGVuX21lbW1hcChzdHIoZmxhZ3NfcCksIG1vZGU9InIrIiBpZiBmbGFnc19wLmV4aXN0cygpIGVsc2UgIncrIiwgZHR5cGU9bnAuYm9vbF8sIHNoYXBlPShuLCkpCiAgICAgICAgdG9kbyA9IFtpIGZvciBpIGluIHJhbmdlKG4pIGlmIG5vdCBmbGFnc1tpXV0KICAgICAgICBwcmludChmImltYWdlczoge24gLSBsZW4odG9kbyl9IGFscmVhZHkgZG9uZSwgZG93bmxvYWRpbmcge2xlbih0b2RvKX0gd2l0aCB7YXJncy50aHJlYWRzfSB0aHJlYWRzIC4uLiIsIGZsdXNoPVRydWUpCiAgICAgICAgdG9yY2guc2V0X251bV90aHJlYWRzKDEpCiAgICAgICAgbG9jaywgY291bnQgPSB0aHJlYWRpbmcuTG9jaygpLCBbMF0KCiAgICAgICAgZGVmIG9uZShpKToKICAgICAgICAgICAgZGF0YSA9IGZldGNoX2J5dGVzKGltYWdlc1tpXVsiY29jb191cmwiXSkKICAgICAgICAgICAgaWYgZGF0YSBpcyBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgICAgIGltZyA9IG5wLmFycmF5KEltYWdlLm9wZW4oaW8uQnl0ZXNJTyhkYXRhKSkuY29udmVydCgiUkdCIikpCiAgICAgICAgICAgIGFycltpXSA9IHNxdWFzaChpbWcsIFMpWzBdLnJvdW5kKCkuYnl0ZSgpLm51bXB5KCkKICAgICAgICAgICAgZmxhZ3NbaV0gPSBUcnVlCiAgICAgICAgICAgIHdpdGggbG9jazoKICAgICAgICAgICAgICAgIGNvdW50WzBdICs9IDEKICAgICAgICAgICAgICAgIGlmIGNvdW50WzBdICUgNTAwMCA9PSAwOgogICAgICAgICAgICAgICAgICAgIGFyci5mbHVzaCgpOyBmbGFncy5mbHVzaCgpCiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiIgIGltYWdlcyB7biAtIGxlbih0b2RvKSArIGNvdW50WzBdfS97bn0gKHt0aW1lLnRpbWUoKSAtIHQwOi4wZn1zKSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIHJldHVybiBUcnVlCgogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDMpOgogICAgICAgICAgICB3aXRoIFRocmVhZFBvb2xFeGVjdXRvcihhcmdzLnRocmVhZHMpIGFzIGV4OgogICAgICAgICAgICAgICAgbGlzdChleC5tYXAob25lLCB0b2RvKSkKICAgICAgICAgICAgYXJyLmZsdXNoKCk7IGZsYWdzLmZsdXNoKCkKICAgICAgICAgICAgdG9kbyA9IFtpIGZvciBpIGluIHJhbmdlKG4pIGlmIG5vdCBmbGFnc1tpXV0KICAgICAgICAgICAgaWYgbm90IHRvZG86CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBwcmludChmIiAge2xlbih0b2RvKX0gaW1hZ2VzIGZhaWxlZCwgcmV0cnlpbmcgLi4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBhc3NlcnQgbm90IHRvZG8sIGYie2xlbih0b2RvKX0gaW1hZ2VzIGNvdWxkIG5vdCBiZSBkb3dubG9hZGVkOiBydW4gdGhlIGNlbGwgYWdhaW4gKGl0IGNvbnRpbnVlcykiCiAgICAgICAgZGVsIGFyciwgZmxhZ3MKICAgICAgICAob3V0IC8gImlucHV0c19jb21wbGV0ZSIpLndyaXRlX3RleHQoIm9rIikKICAgIHByaW50KGYiaW1hZ2VzIHJlYWR5ICh7dGltZS50aW1lKCkgLSB0MDouMGZ9cykiLCBmbHVzaD1UcnVlKQoKICAgICMgNC4gbWFza3MgKyBib3hlcyAoc2FtZSBydWxlcyBhcyBmaW5ldHVuZS5idWlsZF90YXJnZXRzKSwgd3JpdHRlbiBpbiBpbWFnZSBvcmRlcgogICAgcm93c19wLCBtYXNrc19wID0gb3V0IC8gInRhcmdldHNfcm93cy5ucHkiLCBvdXQgLyAidGFyZ2V0c19tYXNrcy5ucHkiCiAgICBpZiBub3Qgcm93c19wLmV4aXN0cygpOgogICAgICAgIGJ5X2ltZyA9IHt9CiAgICAgICAgZm9yIGEgaW4gRFsiYW5ub3RhdGlvbnMiXToKICAgICAgICAgICAgaWYgYVsiaW1hZ2VfaWQiXSBpbiBrZWVwIGFuZCBub3QgYVsiaXNjcm93ZCJdOgogICAgICAgICAgICAgICAgYnlfaW1nLnNldGRlZmF1bHQoYVsiaW1hZ2VfaWQiXSwgW10pLmFwcGVuZCh7InNlZ21lbnRhdGlvbiI6IGFbInNlZ21lbnRhdGlvbiJdfSkKICAgICAgICBHWyJpbWdzIl0gPSBbKGltWyJoZWlnaHQiXSwgaW1bIndpZHRoIl0sIGJ5X2ltZy5nZXQoaW1bImlkIl0sIFtdKSkgZm9yIGltIGluIGltYWdlc10KICAgICAgICB1cHBlciA9IHN1bShsZW4odikgZm9yIHYgaW4gYnlfaW1nLnZhbHVlcygpKQogICAgICAgIGRlbCBELCBieV9pbWcKICAgICAgICBwcmludChmIm1hc2tzOiBkZWNvZGluZyB1cCB0byB7dXBwZXJ9IG9iamVjdHMgd2l0aCB7YXJncy53b3JrZXJzfSBwcm9jZXNzZXMgLi4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBtbSA9IG5wLmxpYi5mb3JtYXQub3Blbl9tZW1tYXAoc3RyKG1hc2tzX3ApLCBtb2RlPSJ3KyIsIGR0eXBlPW5wLnVpbnQ4LCBzaGFwZT0odXBwZXIsIFBBQ0tFRCkpCiAgICAgICAgY2h1bmtzID0gWyhsbywgbWluKG4sIGxvICsgNTAwKSkgZm9yIGxvIGluIHJhbmdlKDAsIG4sIDUwMCldCiAgICAgICAgcm93cywgayA9IFtdLCAwCiAgICAgICAgd2l0aCBQb29sKGFyZ3Mud29ya2VycykgYXMgcG9vbDoKICAgICAgICAgICAgZm9yIGMsIChyLCBwKSBpbiBlbnVtZXJhdGUocG9vbC5pbWFwKG1hc2tzX2Zvcl9jaHVuaywgY2h1bmtzKSk6CiAgICAgICAgICAgICAgICBtbVtrOmsgKyBsZW4ocCldID0gcDsgayArPSBsZW4ocCk7IHJvd3MuYXBwZW5kKHIpCiAgICAgICAgICAgICAgICBpZiAoYyArIDEpICUgMjAgPT0gMCBvciBjICsgMSA9PSBsZW4oY2h1bmtzKToKICAgICAgICAgICAgICAgICAgICBwcmludChmIiAgbWFza3M6IGltYWdlcyB7Y2h1bmtzW2NdWzFdfS97bn0sIG9iamVjdHMge2t9ICh7dGltZS50aW1lKCkgLSB0MDouMGZ9cykiLCBmbHVzaD1UcnVlKQogICAgICAgIG1tLmZsdXNoKCk7IGRlbCBtbQogICAgICAgIHJvd3MgPSBucC5jb25jYXRlbmF0ZShyb3dzKQogICAgICAgIG5wLnNhdmUoc3RyKHJvd3NfcCkgKyAiLnBhcnQubnB5Iiwgcm93cyk7IG9zLnJlcGxhY2Uoc3RyKHJvd3NfcCkgKyAiLnBhcnQubnB5Iiwgcm93c19wKQogICAgcm93cyA9IG5wLmxvYWQocm93c19wKQogICAgZnAgPSBoYXNobGliLnNoYTEobnAubG9hZChvdXQgLyAiaW1hZ2VfaWRzLm5weSIpLnRvYnl0ZXMoKSArIHJvd3MudG9ieXRlcygpKS5oZXhkaWdlc3QoKVs6MTZdCiAgICBpbmZvID0geyJpbWFnZXMiOiBuLCAib2JqZWN0cyI6IGludChsZW4ocm93cykpLCAiaW1hZ2VzX3dpdGhfb2JqZWN0cyI6IGludChsZW4obnAudW5pcXVlKHJvd3NbOiwgMF0pKSksCiAgICAgICAgICAgICJzZWVkIjogYXJncy5zZWVkLCAiZmluZ2VycHJpbnQiOiBmcCwgInNlY29uZHMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHQwKX0KICAgIGpzb24uZHVtcChpbmZvLCBvcGVuKG91dCAvICJwcmVwYXJlXzEwMGtfaW5mby5qc29uIiwgInciKSkKICAgIHByaW50KCJkb25lOiIsIGluZm8sIGZsdXNoPVRydWUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="}
for name, data in FILES.items():
    open(f'{CODE}/repro/{name}', 'wb').write(base64.b64decode(data))

os.environ['EFFSAM_REPO'] = CODE
os.environ['COCO_DATA_ROOT'] = DATA
os.environ['REPRO_OUT'] = f'{WORK}/results/gpu_t4'
os.chdir(CODE)
OUT = f'{WORK}/checkpoints/ft_{INIT}_100k'
print('ready. run:', INIT, '-> checkpoints in', OUT)

In [ ]:
# Cell 2: build the 100,000-image training set on the local disk (~30-60 min). Safe to re-run: it continues.
FEAT = '/content/features_100k'
!python -m repro.prepare_100k --base_subset "{WORK}/features_5k/subset_instances_train2017.json" --out {FEAT}
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'{FEAT}/prepare_100k_info.json', f'{OUT}/prepare_100k_info.json')
print(open(f'{FEAT}/prepare_100k_info.json').read())

In [ ]:
# Cell 3: training, 5 epochs (~3.5-4.5 h). Resumes automatically from the last save in OUT.
enc = f'--encoder "{ENC[INIT]}"' if ENC[INIT] else ''
!python -m repro.finetune --init {INIT} {enc} --features {FEAT} --large --epochs 5 --warmup 0.25 --save_steps 1000 --out "{OUT}"
print('finished:', os.path.exists(f'{OUT}/model_final.pt'))

In [ ]:
# Cell 4: evaluation on the 1,620 held-out objects (~5 min), then download the result file
MODEL = f'ours_{INIT}_100k'
!python -m repro.evaluate_model --model {MODEL} --weights "{OUT}/model_final.pt"
csv = f"{os.environ['REPRO_OUT']}/per_instance_{MODEL}.csv"
print('saved:', csv)
from google.colab import files
files.download(csv)